# The RSA Cryptography Quest: Complete Solution Walkthrough
**Introduction to Number Theory and Cryptography Workshop**  
*Maths and Physics Club, IIT Bombay*

This notebook demonstrates the end-to-end resolution of the workshop's interactive multi-stage cryptographic treasure hunt, connecting all seven cryptanalytic techniques to recover both secret documents and the grand final answer.

In [ ]:
import math

# Core Crypto Primitives
def ConvertToInt(message_str):
    res = 0
    for ch in message_str:
        res = res * 256 + ord(ch)
    return res

def ConvertToStr(n):
    res = ""
    while n > 0:
        res += chr(n % 256)
        n //= 256
    return res[::-1]

def ExtendedEuclid(a, b):
    if b == 0:
        return (1, 0)
    (x, y) = ExtendedEuclid(b, a % b)
    k = a // b
    return (y, x - k * y)

def InvertModulo(a, n):
    (b, x) = ExtendedEuclid(a, n)
    if b < 0:
        b = (b % n + n) % n
    return b

def Encrypt(message, modulo, exponent):
    m = ConvertToInt(message)
    return pow(m, exponent, modulo)

def Decrypt(ciphertext, p, q, exponent):
    phi = (p - 1) * (q - 1)
    d = InvertModulo(exponent, phi)
    m = pow(ciphertext, d, p * q)
    return ConvertToStr(m)

def DecipherSimple(ciphertext, modulo, exponent, potential_messages):
    for msg in potential_messages:
        if Encrypt(msg, modulo, exponent) == ciphertext:
            return msg
    return "don't know"

def DecipherSmallPrime(ciphertext, modulo, exponent):
    p = None
    for i in range(2, 1000001):
        if modulo % i == 0:
            p = i
            break
    q = modulo // p
    return Decrypt(ciphertext, p, q, exponent)

def DecipherSmallDiff(ciphertext, modulo, exponent):
    a = math.isqrt(modulo)
    if a * a < modulo:
        a += 1
    while True:
        b2 = a * a - modulo
        b = math.isqrt(b2)
        if b * b == b2:
            p = a - b
            q = a + b
            break
        a += 1
    return Decrypt(ciphertext, p, q, exponent)

def ChineseRemainderTheorem(n1, r1, n2, r2):
    (x, y) = ExtendedEuclid(n1, n2)
    r = (r1 * n2 * y + r2 * n1 * x) % (n1 * n2)
    return (r + n1 * n2) % (n1 * n2)

def DecipherHastad(first_ciphertext, first_modulo, second_ciphertext, second_modulo):
    C = ChineseRemainderTheorem(first_modulo, first_ciphertext, second_modulo, second_ciphertext)
    m = math.isqrt(C)
    return ConvertToStr(m)

## Stage 1: Cracking the First Secret Link
- **Part 1 (Decryption):** Solved via private key (p, q, e).
- **Part 2 (Simple Decipher):** Intercepted link prefix among URL shorteners.

In [ ]:
p1 = 779849711281
q1 = 748173698927
e1 = 1018651
c1 = 148784435264686331994392
decrypt_first = Decrypt(c1, p1, q1, e1)

c2 = 336184023047118677086739
mod2 = 1110014195838866450995043
exp2 = 767549
shorteners = ["http://goo.gl/", "http://bit.ly/", "http://tinyurl.com/", "http://t.co/", "http://ow.ly/"]
decrypt_second = DecipherSimple(c2, mod2, exp2, shorteners)

secret_link_1 = decrypt_second + decrypt_first
print("Stage 1 Secret Link:", secret_link_1)
# Output: http://bit.ly/2fD8Sdx -> Points to Secret Google Doc 1

## Stage 2: Cracking the Second Secret Link
- Parameters extracted from Secret Google Doc 1
- Part 3 uses DecipherSmallPrime
- Part 4 uses DecipherSmallDiff (Fermat Factorization)

In [ ]:
c_p3 = 2275574988111277110437311214647848935000427550724521671610776681372035260669927879546598212402059626568529572296378786125308417822064565716822176055495534704248368924219660507214955218285229437531331022539446136060045737304497347919927268903882567968402406701512120140117419155759639779776379512631504909857
m_p3 = 4517997441484302732739462818197633745340869919448215356148620255867108411028738486895653782573554891335084161684285442351914740415677262295900238601126392055761648706990593425725625329173683080677363007706828763867611649178136159927562319810830366004181626803874119100160889961902072311468445073289996506931
e_p3 = 1927740953777951565602110467665746890410015037617105427402060961409149865403375973001399584612131360178672080227818127622463053259278787951537329431243
decrypt_third = DecipherSmallPrime(c_p3, m_p3, e_p3)

c_p4 = 17250119573232938159495151836907345788113398812430175654066191733201327934519622049877939234487323106446778939262441322441499883118016470240906983286600405405975275723333283015774857915215814576298063848917112178624770247286048114974808099004729887688547429197942578127510912101233759373809217128865040107305284699411659572170044538767976158851028945020639377891878532187396172245359826784724912031113990729909962190626715090093246089745719249482899605241682396631713155172212615370473237731019162671529937751370463768746592055682933647328743947355593762481102849672215738899019205704001633289554719966
m_p4 = 51719336593283668370261446912126485319255258778617224708410520647959236475322767644636584229070380964088870777607922268179572997129414217601241230215567764098337343256793524076040868978141016236299799031426456716364496058717536559180352016088612210860181989844954906788705998058534134464446292058222426189127398972577715363401457729737792004774929189832678271680193791993183578966529414824171500027009334274604018078392873654591487139190491950796770048687394408709037045383279375259210879772320878973829171531428762995383292175994184457145115155299673171196041635346062296031613203020122378411630001719
e_p4 = 2370692877507893030572035746060036014373915956107619689350920177346027898716147057536168003507966056257828756067470679338655825857154534154205294932307
decrypt_fourth = DecipherSmallDiff(c_p4, m_p4, e_p4)

secret_link_2 = decrypt_third + decrypt_fourth
print("Stage 2 Secret Link:", secret_link_2)
# Output points to Secret Google Doc 2

## Stage 3: The Final Answer Revelation
- Parameters extracted from Secret Google Doc 2
- Solved via DecipherCommonDivisor and DecipherHastad

In [ ]:
c_cd1 = 23402197839825692972957660045778005842149570897525565884030609841447497251897989699441878864356827076382914144503306327825195586612359555424774490982037582237066994974597680950158893991141848342087005739774563510316219462834998856466395854239468748012474813939969736279381474141408975441930226654470509840545902057273419529586319420996953635246047632517133343837614016964611767487664771097138242956384358875235954455242525595951902976992404807998173614628224868047333599482826829214633744976147539512358779840250950906473302037621404671062661398282452008807449216135596254412709706306612180825630285207
m_cd1 = 58430823052528382232133162302686450950325469505875659240893229402912849840784309398807910065914827693927654379087691699343872463431863820528404206238621375034134223224550898236635277014485802812169842923892545064506382750914506131733421755620169771252068722149093463194110389888198706547399113274669324939230289590384817282220066815621515694361776987606124224166139632882923972455465687163365892873407377154186582920232983732264041961425519725867802358413817383859468852644174951734478657652205963797131615641353571742901542970047092056441684257125633460892715564219695534982132772847099500990153159493
e_cd1 = 2746118316073165006639038412489080229254239843347153634155286882078957003180280976570735902213132927273096496684341374829942796710900231180504644399747
c_cd2 = 30727040453983198346085853479092007121170298341280644429200282752975717255888449508484551605881216103173708171060447929559225510082705090077245435032226394315334065553706481613913603534470394670437184947722155153114209268193235042140736923023590768685692783514873340402355775735846109759697748120233777810481952073715032877304902097750445981869076182666622029473694669920172468716779937348673868171294362620903115903645873289564367176851434972937848591902468604400091051094342304122317377572964272370825515688006084797380707045470951321039467067019460709497651682710286878824565888446939229639383746
m_cd2 = 42955660491543430460148980223483714568674234383080724206879432109744068992302351872678094322626198782875395264399683702531442650135501135314224418412956329257244753538669224867463416933578848236691757751760346388773871177654731330626106150400017398309330888903195999995955385444097920846918948184509773583149893677181571427932535786503554494314244294387551141996689565647925375574586590998569956897670722964200467030891957754959187718494336504559943885199686487499306417596091732019643533054349876912582987170110689841980733691588287144182824378328622431439004258091442919814253322774125364502939460173
e_cd2 = 1696307946163562648509752484630349193234799326977939859599277182356207670242676510190455023892125378267322231053560789876259990406520567562708111385869

common_p = math.gcd(m_cd1, m_cd2)
q_cd1 = m_cd1 // common_p
q_cd2 = m_cd2 // common_p
final_part1 = Decrypt(c_cd1, common_p, q_cd1, e_cd1)
final_part2 = Decrypt(c_cd2, common_p, q_cd2, e_cd2)

c_h1 = 1695925818934701958536732606471007147871697773728881305981426383004125888187531096540654444788889942827682032651915812560
m_h1 = 2019703097262944143745237345710626601616589344196612447233041278945420303895066774752152001687355425741826173570952817341
c_h2 = 438180481071044087096048173519586369091293503891810459558098521655659500752405553498472135529822589328291372879622802017
m_h2 = 968914506353797853274340817936111164236562754898105214422095149641604042531040243209810736059746240506447210425767208577
final_part3 = DecipherHastad(c_h1, m_h1, c_h2, m_h2)

grand_final_answer = final_part1 + final_part2 + final_part3
print("=== GRAND FINAL ANSWER ===")
print(grand_final_answer)
assert grand_final_answer == "Congratulations! Now you know that the Answer to the Ultimate Question of Life, the Universe, and Everything is 42."
